# Demostración: cargar, mostrar y reutilizar los modelos entrenados

Este cuaderno muestra que los modelos entrenados **están guardados** y que se pueden **volver a cargar y usar** sin repetir el entrenamiento.

- Los pesos finales (época 100) de las 18 corridas están en la carpeta `modelos/`, un archivo `.pt` por corrida. Se generan con `python exportar_modelos.py` a partir de los puntos de control que dejó el entrenamiento en `results/avance/`.
- Cada archivo guarda los pesos de la red (`net`) y lo necesario para reutilizarla: el tipo de modelo, el número de clases, los nombres de las clases, el experimento, la semilla y la exactitud de prueba reportada.
- Aquí se cargan dos modelos de E2 (ScanObjectNN, 15 clases, semilla 1), se verifica que reproducen la exactitud del informe y se clasifican nubes de puntos nuevas.

Para correrlo hacen falta PyTorch y los datos de ScanObjectNN en `datos/h5_files` (ver el `README.md`). Se ejecutó en la GPU NVIDIA GeForce RTX 3050 del equipo donde se entrenó.

In [1]:
import numpy as np, torch
from torch.utils.data import DataLoader
import data as D
from models import build

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dev, "-", torch.cuda.get_device_name(0) if dev == "cuda" else "CPU")


def cargar_modelo(ruta):
    """Reconstruye la red desde un archivo de modelos/ y le carga los pesos guardados."""
    g = torch.load(ruta, map_location=dev, weights_only=False)
    net = build(g["model"], g["num_classes"]).to(dev)
    net.load_state_dict(g["net"])
    net.eval()
    return net, g

Dispositivo: cuda - NVIDIA GeForce RTX 3050


## 1. Cargar un modelo y mostrar su arquitectura

In [2]:
net, info = cargar_modelo("modelos/sonn_dgcnn_s1.pt")
print({k: v for k, v in info.items() if k not in ("net", "class_names")})
print("Clases:", info["class_names"])
print("Parámetros:", f"{sum(p.numel() for p in net.parameters()) / 1e6:.2f} M")
print()
print(net)

{'model': 'dgcnn', 'num_classes': 15, 'task': 'sonn', 'seed': 1, 'epochs': 100, 'amp': True, 'test_oa': 0.7532963219986121}


Clases: ['bag', 'bin', 'box', 'cabinet', 'chair', 'desk', 'display', 'door', 'shelf', 'table', 'bed', 'pillow', 'sink', 'sofa', 'toilet']
Parámetros: 1.80 M

DGCNN(
  (e1): Sequential(
    (0): Conv2d(6, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
  )
  (e2): Sequential(
    (0): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
  )
  (e3): Sequential(
    (0): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
  )
  (e4): Sequential(
    (0): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track

## 2. Comprobar que el modelo guardado reproduce la exactitud del informe

Se evalúa sobre el conjunto de prueba completo de ScanObjectNN PB_T50_RS (2 882 objetos), con el mismo preprocesamiento del entrenamiento: 1 024 puntos, centrado y escala. DGCNN usa precisión mixta, igual que al entrenar.

In [3]:
Xte, yte, nombres = D.scanobjectnn("datos/h5_files", "test")
loader = DataLoader(D.Clouds(Xte, yte, 1024), 64)


def evaluar(net, usa_amp):
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=usa_amp and dev == "cuda"):
                salida = net(x.to(dev))
            P.append(salida.argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


print(f"{'Modelo':<26}{'OA recalculada':>16}{'OA reportada (results/)':>26}")
for ruta in ["modelos/sonn_dgcnn_s1.pt", "modelos/sonn_pointnet_s1.pt"]:
    m, g = cargar_modelo(ruta)
    P, Y = evaluar(m, g["amp"])
    print(f"{ruta.split('/')[-1][:-3]:<26}{100 * (P == Y).mean():>15.2f} %{100 * g['test_oa']:>24.2f} %")

Modelo                      OA recalculada   OA reportada (results/)


sonn_dgcnn_s1                       75.33 %                   75.33 %


sonn_pointnet_s1                    58.12 %                   58.12 %


## 3. Reutilizar el modelo: clasificar nubes de puntos

La función `clasificar` recibe una nube de puntos (N × 3), la normaliza como en el entrenamiento y devuelve las tres clases más probables. Aquí se prueba con cinco objetos del conjunto de prueba, y se compara con la clase real.

In [4]:
def clasificar(net, nube, clases, npuntos=1024, top=3, usa_amp=True):
    """nube: arreglo (N, 3). Devuelve [(clase, probabilidad), ...] con las `top` más probables."""
    x = D.normalize(nube[None, :, :3].astype(np.float32))[:, :npuntos]
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=usa_amp and dev == "cuda"):
        prob = torch.softmax(net(torch.from_numpy(x).to(dev)).float(), dim=1)[0].cpu().numpy()
    orden = prob.argsort()[::-1][:top]
    return [(clases[i], float(prob[i])) for i in orden]


net, info = cargar_modelo("modelos/sonn_dgcnn_s1.pt")
rng = np.random.RandomState(0)
for i in rng.choice(len(yte), 5, replace=False):
    top = clasificar(net, Xte[i], info["class_names"], usa_amp=info["amp"])
    ok = "acierto" if top[0][0] == nombres[yte[i]] else "error"
    print(f"real: {nombres[yte[i]]:<8} -> predicho: {top[0][0]:<8} ({ok})   top 3:",
          ", ".join(f"{c} {100 * p:.0f}%" for c, p in top))

real: desk     -> predicho: desk     (acierto)   top 3: desk 32%, box 23%, cabinet 20%
real: cabinet  -> predicho: cabinet  (acierto)   top 3: cabinet 92%, sink 5%, desk 1%
real: shelf    -> predicho: shelf    (acierto)   top 3: shelf 85%, door 4%, display 3%
real: toilet   -> predicho: pillow   (error)   top 3: pillow 33%, sink 22%, toilet 8%
real: door     -> predicho: door     (acierto)   top 3: door 95%, cabinet 2%, sofa 1%


## Cómo usar los modelos en otro proyecto

```python
import torch
from models import build            # models.py de este repositorio

g = torch.load("modelos/sonn_dgcnn_s1.pt", map_location="cpu", weights_only=False)
net = build(g["model"], g["num_classes"])
net.load_state_dict(g["net"])
net.eval()
```

Los 18 archivos de `modelos/` pesan en total unos 190 MB y no están en el repositorio por su tamaño.